# Notebook 8: Data Manipulation
**Filename:** `08_Data_Manipulation.ipynb`  
**Topics Covered:** Rename Columns, Drop Rows, Drop Columns, Insert Columns, `assign()`, `apply()`, `map()`, `replace()`, Duplicate Handling

---

## 1. Renaming Columns (`rename`)

### Concept Explanation
`rename()` changes specific column names or index labels using a dictionary mapping (`columns={'old_name': 'new_name'}`). It can operate in-place or return a modified copy.

### Real-world Example
Standardizing raw header names from public APIs to match internal data pipeline naming conventions.

### Business Example
Converting technical column keys like `cust_usr_id_v2` into clean, presentation-ready labels like `Customer_ID` for reporting dashboards.

### AI/ML Example
Renaming raw sensor feature names to match expected input signatures of pretrained model pipelines.


In [1]:
import pandas as pd

# Create base DataFrame
df = pd.DataFrame({
    'emp_id': [101, 102, 103, 104, 104],
    'full_name': [' Alice ', 'Bob', 'Charlie', 'David', 'David'],
    'dept_code': ['HR', 'IT', 'FIN', 'IT', 'IT'],
    'm_salary': [50000, 75000, 62000, 80000, 80000],
    'rating': ['A', 'B', 'A', 'C', 'C']
})

# Rename columns using a dictionary mapping
df_renamed = df.rename(columns={'emp_id': 'Employee_ID', 'm_salary': 'Monthly_Salary'})
print("Renamed Columns:\n", df_renamed.columns.tolist())

Renamed Columns:
 ['Employee_ID', 'full_name', 'dept_code', 'Monthly_Salary', 'rating']


---

## 2. Dropping Rows & Columns (`drop`)

### Concept Explanation
`drop()` removes specified labels from rows (`axis=0`) or columns (`axis=1`). Rows can be dropped by index labels, and columns can be dropped using column names or the `columns` parameter.

### Real-world Example
Removing obsolete metadata columns before saving a cleaned file to disk.

### Business Example
Dropping test/trial transaction rows from production financial accounting tables.

### AI/ML Example
Removing high-cardinality ID columns that contain no predictive signal before feature encoding.

In [2]:
import pandas as pd

# Drop columns (axis=1)
df_no_dept = df_renamed.drop(columns=['dept_code'])

# Drop rows by index labels (axis=0)
df_no_row_0 = df_renamed.drop(index=[0])

print("Columns after dropping 'dept_code':\n", df_no_dept.columns.tolist())
print("\nDataFrame after dropping row 0:\n", df_no_row_0)

Columns after dropping 'dept_code':
 ['Employee_ID', 'full_name', 'Monthly_Salary', 'rating']

DataFrame after dropping row 0:
    Employee_ID full_name dept_code  Monthly_Salary rating
1          102       Bob        IT           75000      B
2          103   Charlie       FIN           62000      A
3          104     David        IT           80000      C
4          104     David        IT           80000      C


---

## 3. Inserting Columns (`insert`)

### Concept Explanation
`insert()` adds a new column into a DataFrame at a specific index position (`loc`). Unlike bracket assignment (`df['new'] = ...`), which adds columns at the end, `insert()` allows precise placement in-place.

### Real-world Example
Placing a calculated status column immediately next to its primary metric column for easier manual visual review.

### Business Example
Inserting a `Country` column right after `State` in demographic contact tables.

### AI/ML Example
Inserting an explicit `Sample_Weight` column at index 0 for downstream training script consumption.

In [3]:
import pandas as pd

df_insert = df_renamed.copy()

# Insert 'Status' at index position 2
df_insert.insert(loc=2, column='Status', value='Active')

print("Inserted Column at Position 2:\n", df_insert.head(2))

Inserted Column at Position 2:
    Employee_ID full_name  Status dept_code  Monthly_Salary rating
0          101    Alice   Active        HR           50000      A
1          102       Bob  Active        IT           75000      B


---

## 4. Method Chaining with `assign()`

### Concept Explanation
`assign()` creates new columns or overwrites existing ones while returning a new DataFrame object. It is ideal for method chaining without altering original source data.

### Real-world Example
Calculating environmental conversion factors step-by-step in a continuous processing pipeline.

### Business Example
Computing annual compensation (`Monthly_Salary * 12`) and bonus estimates in a single chained pipeline.

### AI/ML Example
Generating normalized numerical features during continuous dataset transformation chains.

In [4]:
import pandas as pd

# Method chaining with assign() using lambda functions
df_assigned = df_renamed.assign(
    Annual_Salary=lambda x: x['Monthly_Salary'] * 12,
    Tax_Estimate=lambda x: x['Monthly_Salary'] * 12 * 0.20
)

print("Columns Created via assign():\n", df_assigned[['Monthly_Salary', 'Annual_Salary', 'Tax_Estimate']])

Columns Created via assign():
    Monthly_Salary  Annual_Salary  Tax_Estimate
0           50000         600000      120000.0
1           75000         900000      180000.0
2           62000         744000      148800.0
3           80000         960000      192000.0
4           80000         960000      192000.0


---

## 5. Element-wise & Row-wise Transformations (`apply`)

### Concept Explanation
`apply()` invokes a function across DataFrame axes—either along columns (`axis=0`) or across row elements (`axis=1`).

### Real-world Example
Parsing custom time formats row-by-row using custom logic functions.

### Business Example
Applying custom tier logic across multiple column metrics (e.g., combining Tenure and Spend to set customer tier).

### AI/ML Example
Executing custom string parsing functions on unstructured text columns to construct new feature markers.

In [5]:
import pandas as pd

# Function for row-wise application
def salary_tier(row):
    if row['Monthly_Salary'] > 70000:
        return 'High'
    return 'Standard'

# Applying function along rows (axis=1)
df_assigned['Tier'] = df_assigned.apply(salary_tier, axis=1)

print("Applied Custom Row-wise Function:\n", df_assigned[['Monthly_Salary', 'Tier']])

Applied Custom Row-wise Function:
    Monthly_Salary      Tier
0           50000  Standard
1           75000      High
2           62000  Standard
3           80000      High
4           80000      High


---

## 6. Series Value Mapping (`map`)

### Concept Explanation
`map()` maps values of a Series using a dictionary, Series, or function. Unmapped values in dictionary mappings default to `NaN`.

### Real-world Example
Mapping abbreviated weather station codes (`'NY'`, `'LA'`) to full state titles.

### Business Example
Converting categorical letter ratings (`'A'`, `'B'`, `'C'`) into explicit numeric evaluation scores.

### AI/ML Example
Performing manual label encoding on binary or ordinal categorical features ($Y$ labels).

In [6]:
import pandas as pd

# Rating conversion dictionary
rating_map = {'A': 5, 'B': 4, 'C': 3}

# Map categorical ratings to numeric scores
df_assigned['Rating_Score'] = df_assigned['rating'].map(rating_map)

print("Mapped Rating to Numeric Scores:\n", df_assigned[['rating', 'Rating_Score']])

Mapped Rating to Numeric Scores:
   rating  Rating_Score
0      A             5
1      B             4
2      A             5
3      C             3
4      C             3


---

## 7. Replacing Values (`replace`)

### Concept Explanation
`replace()` swaps target values, lists of values, or regex patterns across Series or DataFrames with replacement values. Unlike `map()`, unreplaced values retain their original values.

### Real-world Example
Cleaning placeholder text entries (like `'N/A'`, `'missing'`, `'-'`) into standard missing values or standard text.

### Business Example
Replacing legacy department names with newly restructured organizational unit names across legacy enterprise logs.

### AI/ML Example
Standardizing sentinel default values (`-999` or `-1`) across numerical feature datasets.

In [7]:
import pandas as pd

# Replace legacy department codes with full names
df_replaced = df_assigned.replace({'HR': 'Human Resources', 'IT': 'Information Tech', 'FIN': 'Finance'})

print("Replaced Values:\n", df_replaced[['Employee_ID', 'dept_code']])

Replaced Values:
    Employee_ID         dept_code
0          101   Human Resources
1          102  Information Tech
2          103           Finance
3          104  Information Tech
4          104  Information Tech


---

## 8. Duplicate Handling (`duplicated`, `drop_duplicates`)

### Concept Explanation
* `duplicated()` identifies repeated rows returning a boolean Series.
* `drop_duplicates()` removes duplicate rows based on all or specified subset columns (`subset`), retaining specified instances (`keep='first'`, `keep='last'`, or `keep=False`).

### Real-world Example
Detecting duplicate form submissions generated by multiple click submissions.

### Business Example
Deduplicating customer accounts using `Employee_ID` to preserve unique user records.

### AI/ML Example
Removing duplicated training instances to prevent dataset leakage between train and test splits.

In [8]:
import pandas as pd

# Identify duplicate rows
print("Duplicate Mask:\n", df_replaced.duplicated())

# Drop duplicates based on specific column subset
df_dedup = df_replaced.drop_duplicates(subset=['Employee_ID'], keep='first')

print("\nShape before deduplication:", df_replaced.shape)
print("Shape after drop_duplicates(subset=['Employee_ID']):", df_dedup.shape)

Duplicate Mask:
 0    False
1    False
2    False
3    False
4     True
dtype: bool

Shape before deduplication: (5, 9)
Shape after drop_duplicates(subset=['Employee_ID']): (4, 9)


---

## Minimum 5 Interview Questions with Answers

1. **How does `df.assign()` differ from standard bracket assignment (`df['col'] = ...`)?**  
   * **Answer:** `df.assign()` returns a new modified DataFrame copy without modifying the original in-place, enabling functional method chaining. Bracket assignment alters the existing DataFrame in-place.

2. **What is the difference between `map()` and `apply()` when modifying a Pandas Series?**  
   * **Answer:** `map()` operates element-wise using dictionaries, functions, or Series (unmapped key entries become `NaN`). `apply()` invokes functions and supports complex row-wise or column-wise operations across multi-column boundaries.

3. **How does `replace()` differ from `map()` when substituting values using a dictionary?**  
   * **Answer:** When passing a dictionary to `map()`, any value not present in the dictionary keys gets replaced with `NaN`. With `replace()`, values not found in the dictionary retain their original values unchanged.

4. **What is the difference between `df.drop(columns=['A'])` and `df.drop(['A'], axis=1)`?**  
   * **Answer:** They perform identical column removals. `columns=['A']` explicitly targets column labels, while `axis=1` tells Pandas to perform label removal along the column dimension.

5. **How does `drop_duplicates(keep=False)` differ from `drop_duplicates(keep='first')`?**  
   * **Answer:** `keep='first'` keeps the first instance of duplicate rows and removes subsequent duplicates. `keep=False` removes all instances of duplicated rows entirely, leaving zero instances in the output.

---

## Self Reflection
* **What I Learned:** Mastered column renaming (`rename`), row/column dropping (`drop`), positioning inserts (`insert`), pipeline creation (`assign`), transformations (`apply`, `map`, `replace`), and deduplication (`drop_duplicates`).
* **Key Takeaway:** Combining clean structural manipulation with functional method chains keeps raw data transformation scripts readable, deterministic, and maintainable.